In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os



In [37]:
load_dotenv()

True

In [38]:
gemini_api_key = os.getenv('gemini_api_key')

In [39]:
# embedding

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",       
    embed_batch_size=100,
    google_api_key=gemini_api_key
)

In [40]:
#llm

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=gemini_api_key,
    temperature=0.7,
    max_output_tokens=1024,
    max_retries=2,
    timeout=60
    
)

In [41]:
# loading data

loader = PyPDFLoader("LangChain.pdf")
docs = loader.load()

In [42]:
# split the data

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
documents=text_splitter.split_documents(docs)
documents[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tools for han-\ndling chat models, integrating retrieval-

In [43]:
# vector db

db = FAISS.from_documents(documents,embedding)

In [44]:
# similarity seach

query="what is langchain "
result=db.similarity_search(query)


In [45]:
print(result[0].page_content)

LangChain
Vasilios Mavroudis
Alan Turing Institute
vmavroudis@turing.ac.uk
Abstract. LangChain is a rapidly emerging framework that offers a ver-
satile and modular approach to developing applications powered by large
language models (LLMs). By leveraging LangChain, developers can sim-
plify complex stages of the application lifecycle—such as development,
productionization, and deployment—making it easier to build scalable,
stateful, and contextually aware applications. It provides tools for han-
dling chat models, integrating retrieval-augmented generation (RAG),
and offering secure API interactions. With LangChain, rapid deployment
of sophisticated LLM solutions across diverse domains becomes feasible.
However, despite its strengths, LangChain’s emphasis on modularity and
integration introduces complexities and potential security concerns that
warrant critical examination. This paper provides an in-depth analysis
of LangChain’s architecture and core components, including LangGraph,


In [46]:
# retrieval

retrieval  = db.as_retriever()

In [47]:
# prompt

prompt = ChatPromptTemplate.from_template("""
Answer the following question using only the information from the provided document.
Think carefully step by step and provide a clear, detailed response.
and only provide direct answer
<context>
{context}
</context>
Question: {input}""")

In [48]:
doc_chain = create_stuff_documents_chain(llm,prompt)

In [49]:
retrieval_chain = create_retrieval_chain(retrieval,doc_chain)

In [50]:
response = retrieval_chain.invoke({"input":"what is langchain"})
response['answer']


'LangChain is a rapidly emerging framework that offers a versatile and modular approach to developing applications powered by large language models (LLMs). It provides tools for handling chat models, integrating retrieval-augmented generation (RAG), and offering secure API interactions.'

In [51]:
import pprint
pp = pprint.PrettyPrinter(indent=5)
pp.pprint(response["answer"])

('LangChain is a rapidly emerging framework that offers a versatile and '
 'modular approach to developing applications powered by large language models '
 '(LLMs). It provides tools for handling chat models, integrating '
 'retrieval-augmented generation (RAG), and offering secure API interactions.')
